<a href="https://colab.research.google.com/github/Chinh2702/KTLT_PYTHONG/blob/gh-pages/2_1_3_B%C3%A0i_t%E1%BA%ADp_th%E1%BB%B1c_h%C3%A0nh_1_X%C3%A2y_d%E1%BB%B1ng_c%C3%A2y_quy%E1%BA%BFt_%C4%91%E1%BB%8Bnh_v%C3%A0_r%E1%BB%ABng_c%C3%A2y_tr%C3%AAn_d%E1%BB%AF_li%E1%BB%87u_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy scipy scikit-learn seaborn matplotlib

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

titanic = sns.load_dataset('titanic')
print("Kích thước dữ liệu:", titanic.shape)
titanic.head()

duplicate_rows = titanic[titanic.duplicated()]
print(f"Số dòng trùng lặp: {duplicate_rows.shape[0]}")

titanic = titanic.drop_duplicates()

print(titanic.info())

for col in titanic.columns:
    print(f"{col}: {titanic[col].unique()[:10]}")

print("Tổng số giá trị thiếu:")
print(titanic.isnull().sum())

titanic['age'] = titanic['age'].fillna(titanic['age'].mean())

titanic['embarked'] = titanic['embarked'].fillna(titanic['embarked'].mode()[0])

titanic = titanic.drop(columns=['deck'])

print("Sau khi xử lý missing value:")
print(titanic.isnull().sum())

num_cols = titanic.select_dtypes(include=[np.number]).columns
print("Các cột số:", num_cols)

for col in num_cols:
    data = titanic[col].dropna()
    print(f"\nCột: {col}")
    print(f"  Giá trị trung bình: {np.mean(data)}")
    print(f"  Trung vị: {np.median(data)}")
    print(f"  Độ lệch chuẩn: {np.std(data)}")
    print(f"  Min: {np.min(data)} - Max: {np.max(data)}")

for col in num_cols:
    data = titanic[col].dropna()
    print(f"\nCột: {col}")
    print(f"  Skewness (độ lệch): {stats.skew(data)}")
    print(f"  Kurtosis (độ nhọn): {stats.kurtosis(data)}")


titanic_encoded = pd.get_dummies(titanic, drop_first=True)

print("Giá trị thiếu còn lại:", titanic_encoded.isnull().sum().sum())

X = titanic_encoded.drop('survived', axis=1)
y = titanic_encoded['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Độ chính xác Decision Tree:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

plt.figure(figsize=(20,10))
plot_tree(dt_model, feature_names=X.columns, class_names=["Không sống sót", "Sống sót"], filled=True)
plt.show()

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Độ chính xác Random Forest:", accuracy_score(y_test, y_pred_rf))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues')
plt.title('Ma trận nhầm lẫn - Random Forest')
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.show()